# Experiment 10: Create Frontend with Real-Time User Input

**Objective:**
- Create a frontend with input forms for real-time predictions
- Show live prediction results
- Integrate with the FastAPI backend

**Prerequisites:** Run Experiments 1-4 (API & model artifacts)

## Step 1: Install Required Libraries

In [ ]:
!pip install streamlit plotly requests pandas ipywidgets

## Step 2: Create Interactive Prediction Form (Jupyter Widgets)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pickle
import pandas as pd
import numpy as np
import requests
import json

# Load model artifacts
with open('model_artifacts/churn_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('model_artifacts/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('model_artifacts/label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)

print("Model loaded! Creating interactive form...")

In [ ]:
# Create form widgets
style = {'description_width': '150px'}
layout = widgets.Layout(width='400px')

gender_widget = widgets.Dropdown(options=['Male', 'Female'], description='Gender:', style=style, layout=layout)
senior_widget = widgets.Dropdown(options=[0, 1], description='Senior Citizen:', style=style, layout=layout)
tenure_widget = widgets.IntSlider(min=0, max=72, value=12, description='Tenure (months):', style=style, layout=layout)
monthly_widget = widgets.FloatSlider(min=0, max=200, value=50.0, step=0.5, description='Monthly Charges:', style=style, layout=layout)
contract_widget = widgets.Dropdown(options=['Month-to-month', 'One year', 'Two year'], description='Contract:', style=style, layout=layout)
payment_widget = widgets.Dropdown(
    options=['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'],
    description='Payment Method:', style=style, layout=layout
)
total_widget = widgets.FloatText(value=600.0, description='Total Charges:', style=style, layout=layout)

predict_button = widgets.Button(description='🔮 Predict Churn', button_style='primary', 
                                 layout=widgets.Layout(width='400px', height='40px'))
output = widgets.Output()

def on_predict(b):
    with output:
        clear_output()
        
        input_data = {
            'Gender': gender_widget.value,
            'SeniorCitizen': senior_widget.value,
            'Tenure': tenure_widget.value,
            'MonthlyCharges': monthly_widget.value,
            'Contract': contract_widget.value,
            'PaymentMethod': payment_widget.value,
            'TotalCharges': total_widget.value
        }
        
        # Try API first, fallback to local model
        try:
            response = requests.post(
                "http://localhost:8000/predict",
                json=input_data,
                headers={"X-API-Key": "mlops-api-key-001"},
                timeout=5
            )
            result = response.json()
            source = "API"
        except:
            # Local prediction
            df_input = pd.DataFrame([input_data])
            for col in label_encoders:
                if col in df_input.columns:
                    df_input[col] = label_encoders[col].transform(df_input[col])
            scaled = scaler.transform(df_input)
            pred = model.predict(scaled)[0]
            proba = model.predict_proba(scaled)[0]
            result = {
                'prediction': int(pred),
                'prediction_label': 'Churn' if pred == 1 else 'No Churn',
                'churn_probability': round(float(proba[1]), 4),
                'no_churn_probability': round(float(proba[0]), 4)
            }
            source = "Local Model"
        
        # Display results
        churn_prob = result.get('churn_probability', 0)
        is_churn = result.get('prediction', 0) == 1
        color = '#e74c3c' if is_churn else '#2ecc71'
        icon = '⚠️' if is_churn else '✅'
        
        html = f"""
        <div style="padding:20px; border-radius:10px; background:{color}20; border:2px solid {color}; margin:10px 0;">
            <h2 style="color:{color}; margin:0;">{icon} Prediction: {result.get('prediction_label', 'N/A')}</h2>
            <hr style="border-color:{color};">
            <table style="width:100%; font-size:16px;">
                <tr><td><b>Churn Probability:</b></td><td>{churn_prob:.2%}</td></tr>
                <tr><td><b>No Churn Probability:</b></td><td>{result.get('no_churn_probability', 0):.2%}</td></tr>
                <tr><td><b>Source:</b></td><td>{source}</td></tr>
            </table>
            <div style="margin-top:10px; background:#ddd; border-radius:5px; height:30px;">
                <div style="background:{color}; width:{churn_prob*100}%; height:30px; border-radius:5px; text-align:center; line-height:30px; color:white; font-weight:bold;">
                    {churn_prob:.1%}
                </div>
            </div>
        </div>
        <div style="padding:10px; background:#f8f9fa; border-radius:5px; margin-top:10px;">
            <b>Input:</b> {json.dumps(input_data, indent=2)}
        </div>
        """
        display(HTML(html))

predict_button.on_click(on_predict)

# Display form
form_title = widgets.HTML('<h2>🏦 Bank Customer Churn Prediction</h2><p>Enter customer details below:</p>')
form = widgets.VBox([
    form_title,
    gender_widget, senior_widget, tenure_widget,
    monthly_widget, contract_widget, payment_widget,
    total_widget, predict_button, output
])
display(form)

## Step 3: Create Full Streamlit App with Real-Time Input

In [ ]:
streamlit_app = '''import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import requests
import pickle
import json

st.set_page_config(page_title="Churn Predictor", page_icon="🏦", layout="wide")

# Load model
@st.cache_resource
def load_model():
    with open("model_artifacts/churn_model.pkl", "rb") as f:
        model = pickle.load(f)
    with open("model_artifacts/scaler.pkl", "rb") as f:
        scaler = pickle.load(f)
    with open("model_artifacts/label_encoders.pkl", "rb") as f:
        encoders = pickle.load(f)
    return model, scaler, encoders

model, scaler, encoders = load_model()

# Sidebar
st.sidebar.title("🏦 Bank Churn Predictor")
st.sidebar.markdown("---")
mode = st.sidebar.radio("Mode", ["Single Prediction", "Batch Prediction", "Dashboard"])
use_api = st.sidebar.checkbox("Use API (requires server running)", value=False)
api_url = st.sidebar.text_input("API URL", "http://localhost:8000")
api_key = st.sidebar.text_input("API Key", "mlops-api-key-001", type="password")

def predict_local(data):
    df = pd.DataFrame([data])
    for col in encoders:
        if col in df.columns:
            df[col] = encoders[col].transform(df[col])
    scaled = scaler.transform(df)
    pred = model.predict(scaled)[0]
    proba = model.predict_proba(scaled)[0]
    return {"prediction": int(pred), "prediction_label": "Churn" if pred == 1 else "No Churn",
            "churn_probability": round(float(proba[1]), 4), "no_churn_probability": round(float(proba[0]), 4)}

def predict_api(data):
    try:
        resp = requests.post(f"{api_url}/predict", json=data, headers={"X-API-Key": api_key}, timeout=5)
        return resp.json()
    except:
        st.warning("API unreachable, using local model")
        return predict_local(data)

if mode == "Single Prediction":
    st.title("🔮 Single Customer Prediction")
    st.markdown("Enter customer information to predict churn risk")
    
    col1, col2 = st.columns(2)
    with col1:
        gender = st.selectbox("Gender", ["Male", "Female"])
        senior = st.selectbox("Senior Citizen", [0, 1])
        tenure = st.slider("Tenure (months)", 0, 72, 12)
        monthly = st.number_input("Monthly Charges ($)", 0.0, 200.0, 50.0, step=0.5)
    with col2:
        contract = st.selectbox("Contract", ["Month-to-month", "One year", "Two year"])
        payment = st.selectbox("Payment", ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"])
        total = st.number_input("Total Charges ($)", 0.0, 10000.0, 600.0, step=10.0)
    
    if st.button("🔮 Predict", type="primary", use_container_width=True):
        data = {"Gender": gender, "SeniorCitizen": senior, "Tenure": tenure,
                "MonthlyCharges": monthly, "Contract": contract,
                "PaymentMethod": payment, "TotalCharges": total}
        
        result = predict_api(data) if use_api else predict_local(data)
        
        col1, col2, col3 = st.columns(3)
        is_churn = result["prediction"] == 1
        col1.metric("Prediction", result["prediction_label"], delta="High Risk" if is_churn else "Low Risk", delta_color="inverse")
        col2.metric("Churn Probability", f"{result['churn_probability']:.2%}")
        col3.metric("Retention Probability", f"{result['no_churn_probability']:.2%}")
        
        # Gauge chart
        fig = go.Figure(go.Indicator(
            mode="gauge+number+delta",
            value=result["churn_probability"] * 100,
            title={"text": "Churn Risk Score"},
            delta={"reference": 50},
            gauge={"axis": {"range": [0, 100]},
                   "bar": {"color": "#e74c3c" if is_churn else "#2ecc71"},
                   "steps": [{"range": [0, 30], "color": "#d4edda"}, {"range": [30, 70], "color": "#fff3cd"}, {"range": [70, 100], "color": "#f8d7da"}],
                   "threshold": {"line": {"color": "red", "width": 4}, "thickness": 0.75, "value": 50}}
        ))
        fig.update_layout(height=300)
        st.plotly_chart(fig, use_container_width=True)

elif mode == "Batch Prediction":
    st.title("📊 Batch Prediction")
    uploaded = st.file_uploader("Upload CSV file", type="csv")
    if uploaded:
        batch_df = pd.read_csv(uploaded)
        st.dataframe(batch_df.head())
        if st.button("Predict All"):
            results = []
            progress = st.progress(0)
            for i, row in batch_df.iterrows():
                data = {"Gender": row.get("Gender", "Male"), "SeniorCitizen": int(row.get("SeniorCitizen", 0)),
                        "Tenure": int(row.get("Tenure", 0)), "MonthlyCharges": float(row.get("MonthlyCharges", 0)),
                        "Contract": row.get("Contract", "Month-to-month"),
                        "PaymentMethod": row.get("PaymentMethod", "Electronic check"),
                        "TotalCharges": float(row.get("TotalCharges", 0))}
                results.append(predict_local(data))
                progress.progress((i+1)/len(batch_df))
            results_df = pd.DataFrame(results)
            batch_df = pd.concat([batch_df, results_df], axis=1)
            st.dataframe(batch_df)
            st.download_button("Download Results", batch_df.to_csv(index=False), "predictions.csv")

else:  # Dashboard
    st.title("📈 Churn Analytics Dashboard")
    df = pd.read_csv("Bank_Churn_Classification_Dataset.csv", index_col=0)
    df_proc = df.drop("CustomerID", axis=1).copy()
    for col in encoders:
        if col in df_proc.columns:
            df_proc[col] = encoders[col].transform(df_proc[col])
    X = df_proc.drop("Churn", axis=1)
    df["Churn_Prob"] = model.predict_proba(scaler.transform(X))[:, 1]
    df["Predicted"] = model.predict(scaler.transform(X))
    
    c1, c2 = st.columns(2)
    with c1:
        fig = px.pie(df, names=df["Predicted"].map({0: "Stay", 1: "Churn"}), title="Predictions")
        st.plotly_chart(fig, use_container_width=True)
    with c2:
        fig = px.histogram(df, x="Churn_Prob", nbins=50, title="Probability Distribution")
        st.plotly_chart(fig, use_container_width=True)
    
    st.subheader("High-Risk Customers")
    st.dataframe(df.nlargest(20, "Churn_Prob")[["CustomerID", "Gender", "Tenure", "MonthlyCharges", "Contract", "Churn_Prob"]])
'''

with open('frontend_app.py', 'w') as f:
    f.write(streamlit_app)

print("Streamlit app created: frontend_app.py")
print("Run with: streamlit run frontend_app.py")

## Step 4: Create HTML Frontend (Standalone)

In [ ]:
html_frontend = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Bank Churn Predictor</title>
    <style>
        * { box-sizing: border-box; margin: 0; padding: 0; font-family: 'Segoe UI', sans-serif; }
        body { background: #f0f2f5; min-height: 100vh; display: flex; justify-content: center; padding: 20px; }
        .container { max-width: 800px; width: 100%; }
        h1 { text-align: center; color: #2c3e50; margin: 20px 0; font-size: 2em; }
        .card { background: white; border-radius: 12px; padding: 30px; margin: 20px 0; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }
        .form-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }
        .form-group { display: flex; flex-direction: column; }
        label { font-weight: 600; color: #555; margin-bottom: 5px; font-size: 14px; }
        select, input { padding: 10px; border: 2px solid #ddd; border-radius: 8px; font-size: 16px; }
        select:focus, input:focus { border-color: #3498db; outline: none; }
        .btn { width: 100%; padding: 15px; background: #3498db; color: white; border: none; border-radius: 8px;
               font-size: 18px; cursor: pointer; margin-top: 20px; font-weight: 600; }
        .btn:hover { background: #2980b9; }
        .result { margin-top: 20px; padding: 20px; border-radius: 12px; text-align: center; }
        .result.churn { background: #fde8e8; border: 2px solid #e74c3c; }
        .result.no-churn { background: #e8fde8; border: 2px solid #2ecc71; }
        .result h2 { font-size: 24px; margin-bottom: 10px; }
        .prob-bar { height: 30px; background: #ecf0f1; border-radius: 15px; overflow: hidden; margin: 10px 0; }
        .prob-fill { height: 100%; border-radius: 15px; transition: width 0.5s; display: flex; align-items: center;
                     justify-content: center; color: white; font-weight: bold; }
        .hidden { display: none; }
        .api-config { font-size: 13px; color: #888; margin-bottom: 10px; }
        .api-config input { font-size: 13px; padding: 5px; width: 250px; }
    </style>
</head>
<body>
    <div class="container">
        <h1>🏦 Bank Customer Churn Predictor</h1>
        <div class="card">
            <div class="api-config">
                API: <input id="apiUrl" value="http://localhost:8000" />
                Key: <input id="apiKey" value="mlops-api-key-001" type="password" />
            </div>
            <div class="form-grid">
                <div class="form-group">
                    <label>Gender</label>
                    <select id="gender"><option>Male</option><option>Female</option></select>
                </div>
                <div class="form-group">
                    <label>Senior Citizen</label>
                    <select id="senior"><option value="0">No (0)</option><option value="1">Yes (1)</option></select>
                </div>
                <div class="form-group">
                    <label>Tenure (months)</label>
                    <input type="number" id="tenure" value="12" min="0" max="72">
                </div>
                <div class="form-group">
                    <label>Monthly Charges ($)</label>
                    <input type="number" id="monthly" value="50" min="0" step="0.5">
                </div>
                <div class="form-group">
                    <label>Contract</label>
                    <select id="contract"><option>Month-to-month</option><option>One year</option><option>Two year</option></select>
                </div>
                <div class="form-group">
                    <label>Payment Method</label>
                    <select id="payment"><option>Electronic check</option><option>Mailed check</option><option>Bank transfer (automatic)</option><option>Credit card (automatic)</option></select>
                </div>
                <div class="form-group" style="grid-column: span 2;">
                    <label>Total Charges ($)</label>
                    <input type="number" id="total" value="600" min="0" step="10">
                </div>
            </div>
            <button class="btn" onclick="predict()">🔮 Predict Churn</button>
        </div>
        <div id="result" class="hidden"></div>
    </div>
    <script>
        async function predict() {
            const apiUrl = document.getElementById('apiUrl').value;
            const apiKey = document.getElementById('apiKey').value;
            const data = {
                Gender: document.getElementById('gender').value,
                SeniorCitizen: parseInt(document.getElementById('senior').value),
                Tenure: parseInt(document.getElementById('tenure').value),
                MonthlyCharges: parseFloat(document.getElementById('monthly').value),
                Contract: document.getElementById('contract').value,
                PaymentMethod: document.getElementById('payment').value,
                TotalCharges: parseFloat(document.getElementById('total').value)
            };
            try {
                const resp = await fetch(`${apiUrl}/predict`, {
                    method: 'POST', headers: {'Content-Type': 'application/json', 'X-API-Key': apiKey},
                    body: JSON.stringify(data)
                });
                const result = await resp.json();
                const isChurn = result.prediction === 1;
                const prob = (result.churn_probability * 100).toFixed(1);
                const color = isChurn ? '#e74c3c' : '#2ecc71';
                document.getElementById('result').className = `card result ${isChurn ? 'churn' : 'no-churn'}`;
                document.getElementById('result').innerHTML = `
                    <h2 style="color:${color}">${isChurn ? '⚠️' : '✅'} ${result.prediction_label}</h2>
                    <p>Churn Probability: <strong>${prob}%</strong></p>
                    <div class="prob-bar"><div class="prob-fill" style="width:${prob}%; background:${color}">${prob}%</div></div>
                    <p style="color:#888; margin-top:10px">No Churn: ${(result.no_churn_probability * 100).toFixed(1)}%</p>
                `;
            } catch (e) {
                document.getElementById('result').className = 'card result churn';
                document.getElementById('result').innerHTML = `<h2>❌ Error</h2><p>${e.message}. Is the API server running?</p>`;
            }
        }
    </script>
</body>
</html>
'''

with open('frontend.html', 'w') as f:
    f.write(html_frontend)

print("HTML frontend created: frontend.html")
print("Open in browser to use the prediction form.")
print("\n✅ Real-time input frontend completed!")

## Step 5: Test the Interactive Widget

In [ ]:
# Quick test - simulate a prediction
test_cases = [
    {"Gender": "Male", "SeniorCitizen": 0, "Tenure": 60, "MonthlyCharges": 50.0,
     "Contract": "Two year", "PaymentMethod": "Bank transfer (automatic)", "TotalCharges": 3000.0},
    {"Gender": "Female", "SeniorCitizen": 1, "Tenure": 1, "MonthlyCharges": 100.0,
     "Contract": "Month-to-month", "PaymentMethod": "Electronic check", "TotalCharges": 100.0},
]

print("Quick prediction test:")
for i, case in enumerate(test_cases):
    df_input = pd.DataFrame([case])
    for col in label_encoders:
        if col in df_input.columns:
            df_input[col] = label_encoders[col].transform(df_input[col])
    scaled = scaler.transform(df_input)
    pred = model.predict(scaled)[0]
    proba = model.predict_proba(scaled)[0]
    print(f"  Case {i+1}: {'Churn' if pred == 1 else 'No Churn'} (prob: {proba[1]:.4f})")

print("\nFiles created:")
print("  - frontend_app.py (Streamlit - run: streamlit run frontend_app.py)")
print("  - frontend.html (HTML - open in browser)")